In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
import re

headers = {"User-Agent": "Mozilla/5.0"}

city_areas = {
        "bangalore": ["indiranagar", "whitefield", "koramangala", "btm", "hsr"],
    "chennai": ["t-nagar", "velachery", "adyar", "porur", "nungambakkam"],
    "varanasi": ['lanka', 'sigra', 'bhelupur',  'assi-ghat', 'chetganj', 'nadesar', 'dashaswmedh-road', 'mahmoorganj'],
    "trivandrum": ['kannanthura', 'kazhakkoottam', 'ambalamukku', 'kesavadasapuram', 'kumarapuram', 'palayam', 'thycaud', 'kulathoor'],
    "thrissur": ['poothole', 'ayyanthole', 'guruvayur-locality', 'chalakudy', 'irinjalakuda-locality', 'peringavu', 'kizhakkumpattukara', 'puzhakkal'],
    "pune": ['kothrud', 'wakad', 'hinjawadi', 'kalyani-nagar', 'kharadi','viman-nagar'],
    "puducherry": ['auroville', 'gandhinagar', 'mudaliarpet', 'kottukuppam', 'heritage-town', 'lawspet', 'white-town', 'mg-road'],
    "ooty":['anna-nagar', 'kandal', 'marlimund', 'elk-hill', 'fern-hill', 'thalayathimund'],
    "mysore": ['vijay-nagar', 'chamrajpura', 'doora', 'mandi-mohalla', 'jayalakhsmipuram', 'bannimantap', 'kuvempunagar', 'gokulam'],
    "manali": ['old-manali','aleo', 'tibetan-colony'],
    "mangalore": ['lalbagh', 'balmatta', 'kodailbail', 'hampankatta', 'kadri', 'ks-rao-nagar',  'kankanady', 'attavar'],
    "madurai": ['kk-nagar', 'arrapalayam', 'iyer-bungalow', 'viswanathapuram', 'periyar', 'othakadai', 'ss-colony', 'sathamangalam'],
    "lucknow": ['gomti-nagar', 'hazratganj', 'aliganj', 'aashiana', 'aminabad', 'indira-nagar', 'alambagh', 'chowk'],
    "kolkata": ['park-street-area', 'sector-1-salt-lake', 'ballygunge', 'camac-street-area', 'southern-avenue', 'new-town', 'sector-5-salt-lake', 'chinar-park'],
    "goa": ['calangute', 'margao', 'vagator', 'panaji', 'arambol', 'anjuna', 'baga', 'candolim'],
    "delhi": ['connaught-place-delhi', 'rajouri-garden-delhi', 'sector-18-noida', 'golf-course-road', 'dlf-cyber-city',  'sector-29-gurgaon-gurugram', 'saket-delhi', 'dlf-phase-4'],
    "coimbatore": ['ramanathapuram', 'town-hall', 'kalapatti', 'peelamedu',  'rs-puram', 'saibaba-colony', 'gandhipuram', 'race-course'],
    "chennai": ['thuraipakkam', 'mylapore', 'alwarpet', 't-nagar', 'adyar', 'velachery', 'nungambakkam', 'anna-nagar-east'],
    "amritsar": ['town-hall', 'basant-nagar',  'ina-colony',  'lawrence-road', 'white-avenue', 'gt-road', 'kabir-park', 'ranjit-avenue'],
    "agra": ['rakabganj', 'khandari', 'tajganj', 'kamla-nagar', 'agra-cantt', 'sikandra', 'civil-lines', 'mantola'],
    "kochi":['edappally', 'panampilly-nagar', 'kacheripady', 'vyttila',  'fort-kochi', 'palarivattom', 'kakkanad', 'mg-road'],
    "jamshedpur": ['telco-colony', 'golmuri', 'sakchi', 'bistupur', 'gamharia', 'sonari', 'mango', 'birsanagar']

}


data = []

# -----------------------------
# SWIGGY PROXY
# -----------------------------
def get_swiggy_proxy(area, city):
    try:
        url = f"https://www.swiggy.com/{city}/{area}"
        res = requests.get(url, headers=headers, timeout=10)
        return len(res.text)
    except:
        return None

# WIKIPEDIA
# -----------------------------
def get_wiki(city):
    try:
        url = f"https://en.wikipedia.org/wiki/{city.capitalize()}"
        res = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(res.text, "lxml")
        text = soup.get_text()

        pop = re.search(r'Population.*?(\d{1,3}(?:,\d{3})+)', text)
        area = re.search(r'Area.*?(\d{1,3}(?:,\d{3})*)\s?km', text)

        return (
            pop.group(1) if pop else None,
            area.group(1) if area else None,
            "yes" if "tourism" in text.lower() else "no",
            "yes" if "gdp" in text.lower() else "no"
        )
    except:
        return None, None, None, None

# -----------------------------
# MAIN SCRAPER
# ----------------------#

for city in city_areas:

    print(f"\nCITY: {city}")
    city_population,city_area_km, tourism_flag, gdp_flag = get_wiki(city)

    for area in city_areas[city]:

        swiggy_proxy = get_swiggy_proxy(area, city)

        for page in range(1, 11):

            url = f"https://www.zomato.com/{city}/{area}-restaurants?page={page}"
            print(f"{city}-{area} Page {page} | Rows: {len(data)}")

            try:
                res = requests.get(url, headers=headers, timeout=10)

                if res.status_code != 200:
                    break

                soup = BeautifulSoup(res.text, "lxml")
                cards = soup.find_all("div", class_="jumbo-tracker")

                if not cards:
                    break

                for card in cards:
                    try:
                        name = card.find("h4")
                        name = name.text.strip() if name else None

                        cuisine = card.find("p")
                        cuisine = cuisine.text.strip() if cuisine else None

                        price_tag = card.find(string=lambda t: t and "₹" in t)
                        price_text = price_tag.strip() if price_tag else None

                        cost = None
                        if price_text:
                            match = re.search(r'\d+', price_text.replace(',', ''))
                            if match:
                                cost = int(match.group())

                        rating_tag = card.find(string=lambda x: x and re.match(r'^\d\.\d$', x))
                        rating = float(rating_tag) if rating_tag else None


                        text_blob = card.get_text().lower()

                        restaurant_type = "casual" if "casual" in text_blob else "quick_bite"
                        is_pure_veg = 1 if "pure veg" in text_blob else 0
                        delivery_available = 1 if "delivery" in text_blob else 0
                        dinein_available = 1 if "dine" in text_blob else 0

                        cuisine_count = len(cuisine.split(",")) if cuisine else 0

                        if cost:
                            if cost < 200:
                                price_category = "low"
                            elif cost < 500:
                                price_category = "medium"
                            else:
                                price_category = "high"
                        else:
                            price_category = None

                        if rating:
                            if rating >= 4:
                                rating_category = "excellent"
                            elif rating >= 3:
                                rating_category = "good"
                            else:
                                rating_category = "poor"
                        else:
                            rating_category = None


                        link_tag = card.find("a", href=True)
                        link = "https://www.zomato.com" + link_tag['href'] if link_tag else None

                        text_blob = card.get_text().lower()
                        order_online = "yes" if "order" in text_blob else "no"
                        book_table = "yes" if "book" in text_blob else "no"



                        if name:
                            data.append([
                                name, cuisine, price_text, cost, rating,
                                city, area,
                                restaurant_type,
                                is_pure_veg,
                                delivery_available,
                                dinein_available,
                                cuisine_count,
                                price_category,
                                rating_category,
                                swiggy_proxy,
                                area_population,
                                tourism_flag,
                                link,
                                city_population,
                                city_area_km,
                                gdp_flag,
                                order_online,
                                book_table,
                            ])

                    except:
                        continue

                time.sleep(random.uniform(0.5, 1.2))

            except:
                break


# -----------------------------
# DATAFRAME
# -----------------------------
columns = [
    "name","cuisine","price_text","cost","rating",
    "city","area",
    "restaurant_type",
    "is_pure_veg",
    "delivery_available",
    "dinein_available",
    "cuisine_count",
    "price_category",
    "rating_category",
    "swiggy_proxy",
    "area_population",
    "tourism_flag",
    "restaurant_url",
    "city_population",
    "city_area_km",
    "gdp_flag",
    "order_online",
    "book_table"
]

df = pd.DataFrame(data, columns=columns)

df.drop_duplicates(inplace=True)

df.to_csv("enhanced_dataset.csv", index=False)

print("\n✅ DONE! Rows:", len(df))


CITY: bangalore
bangalore-indiranagar Page 1 | Rows: 0
bangalore-indiranagar Page 2 | Rows: 9
bangalore-indiranagar Page 3 | Rows: 18
bangalore-indiranagar Page 4 | Rows: 27
bangalore-indiranagar Page 5 | Rows: 36
bangalore-indiranagar Page 6 | Rows: 45
bangalore-indiranagar Page 7 | Rows: 54
bangalore-indiranagar Page 8 | Rows: 63
bangalore-indiranagar Page 9 | Rows: 72
bangalore-indiranagar Page 10 | Rows: 81
bangalore-whitefield Page 1 | Rows: 90
bangalore-whitefield Page 2 | Rows: 99
bangalore-whitefield Page 3 | Rows: 108
bangalore-whitefield Page 4 | Rows: 117
bangalore-whitefield Page 5 | Rows: 126
bangalore-whitefield Page 6 | Rows: 135
bangalore-whitefield Page 7 | Rows: 144
bangalore-whitefield Page 8 | Rows: 153
bangalore-whitefield Page 9 | Rows: 162
bangalore-whitefield Page 10 | Rows: 171
bangalore-koramangala Page 1 | Rows: 180
bangalore-koramangala Page 2 | Rows: 189
bangalore-btm Page 1 | Rows: 189
bangalore-btm Page 2 | Rows: 198
bangalore-btm Page 3 | Rows: 207
bang